# 台风预测模型训练与评估 Notebook

## 文档信息
- **版本**: v1.0
- **创建日期**: 2026-02-23
- **模型架构**: Transformer + LSTM 混合神经网络
- **预测目标**: 台风路径（纬度、经度）与强度（气压、风速）

## 目录
1. [环境配置与依赖安装](#section1)
2. [数据准备与探索](#section2)
3. [模型架构定义](#section3)
4. [模型训练](#section4)
5. [模型评估](#section5)
6. [结果可视化](#section6)
7. [模型保存与导出](#section7)

---

## 1. 环境配置与依赖安装 <a id='section1'></a>

### 1.1 系统环境检查

In [ ]:
# 检查Python版本
import sys
print(f'Python版本: {sys.version}')
print(f'Python路径: {sys.executable}')

# 检查操作系统
import platform
print(f'\n操作系统: {platform.system()} {platform.release()}')
print(f'机器架构: {platform.machine()}')

In [ ]:
# 检查CUDA可用性（GPU训练必需）
import torch

print('=' * 60)
print('PyTorch与CUDA环境检查')
print('=' * 60)
print(f'PyTorch版本: {torch.__version__}')
print(f'CUDA可用: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'CUDA版本: {torch.version.cuda}')
    print(f'GPU数量: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
        print(f'    显存: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB')
else:
    print('警告: CUDA不可用，将使用CPU进行训练（速度较慢）')

# 设置默认设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\n默认设备: {device}')

### 1.2 安装依赖包

In [ ]:
# 核心依赖安装
# 如果已安装可以跳过此步骤

import subprocess
import sys

required_packages = [
    'torch==2.6.0',
    'numpy==2.1.3',
    'pandas==2.2.3',
    'scikit-learn==1.5.2',
    'tqdm>=4.65.0',
    'matplotlib>=3.7.0',
    'seaborn>=0.12.0',
    'plotly>=5.15.0'
]

print('检查并安装依赖包...')
print('=' * 60)

for package in required_packages:
    try:
        # 尝试导入包
        pkg_name = package.split('==')[0].split('>=')[0]
        __import__(pkg_name.replace('-', '_'))
        print(f'✓ {pkg_name} 已安装')
    except ImportError:
        print(f'✗ {pkg_name} 未安装，正在安装...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
        print(f'  ✓ {pkg_name} 安装完成')

print('\n所有依赖包检查完成！')

In [ ]:
# 导入所有必要的库
import os
import sys
import json
import logging
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from typing import List, Optional, Tuple, Dict, Any

# 科学计算
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

# 可视化
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Circle
from matplotlib.lines import Line2D

# 进度条 - 使用标准tqdm避免ipywidgets依赖
from tqdm import tqdm

# 设置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# 设置中文字体显示 - 强制重建字体缓存
import matplotlib.font_manager as fm
import os

# 清除字体缓存并重建
cache_dir = os.path.expanduser('~/.matplotlib')
if os.path.exists(cache_dir):
    for f in os.listdir(cache_dir):
        if f.startswith('fontlist'):
            try:
                os.remove(os.path.join(cache_dir, f))
                print(f'已清除字体缓存: {f}')
            except:
                pass

# 重建字体管理器
fm._load_fontmanager(try_read_cache=False)

# 查找可用的中文字体
def get_chinese_font():
    chinese_fonts = ['SimHei', 'Microsoft YaHei', 'STHeiti', 'SimSun', 
                    'Noto Sans SC', 'KaiTi', 'FangSong', 'STKaiti']
    available_fonts = [f.name for f in fm.fontManager.ttflist]
    print(f'检测到 {len(set(available_fonts))} 种字体')
    for font in chinese_fonts:
        if font in available_fonts:
            print(f'找到中文字体: {font}')
            return font
    return 'DejaVu Sans'

chinese_font = get_chinese_font()
print(f'使用中文字体: {chinese_font}')

# 设置全局字体
plt.rcParams['font.sans-serif'] = [chinese_font]
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# 设置美观的样式
sns.set_style('whitegrid')
sns.set_palette('husl')
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('ggplot')

# 定义配色方案
COLORS = {
    'primary': '#3498db',
    'secondary': '#e74c3c',
    'success': '#2ecc71',
    'warning': '#f39c12',
    'info': '#9b59b6',
    'dark': '#34495e',
    'light': '#ecf0f1'
}

print('✓ 所有库导入成功！')


### 1.3 项目路径配置

In [ ]:
# 配置项目路径
# 获取当前Notebook所在目录
NOTEBOOK_DIR = Path.cwd()
BACKEND_DIR = NOTEBOOK_DIR.parent
PROJECT_ROOT = BACKEND_DIR.parent

# 数据路径
DATA_DIR = BACKEND_DIR / 'data' / 'csv'
CSV_PATH = DATA_DIR / 'typhoon_paths_1966_2026.csv'

# 模型保存路径
MODELS_DIR = NOTEBOOK_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# 结果输出路径
RESULTS_DIR = NOTEBOOK_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# 添加backend到Python路径
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

print('项目路径配置:')
print('=' * 60)
print(f'Notebook目录: {NOTEBOOK_DIR}')
print(f'Backend目录: {BACKEND_DIR}')
print(f'项目根目录: {PROJECT_ROOT}')
print(f'数据目录: {DATA_DIR}')
print(f'CSV文件: {CSV_PATH}')
print(f'模型保存目录: {MODELS_DIR}')
print(f'结果输出目录: {RESULTS_DIR}')
print(f'\nPython路径已添加: {BACKEND_DIR}')

# 验证CSV文件存在
if CSV_PATH.exists():
    print(f'\n✓ CSV数据文件存在')
    print(f'  文件大小: {CSV_PATH.stat().st_size / 1024 / 1024:.2f} MB')
else:
    print(f'\n✗ CSV数据文件不存在: {CSV_PATH}')
    print('  请检查数据文件路径')

---

## 2. 数据准备与探索 <a id='section2'></a>

### 2.1 加载CSV数据

In [ ]:
# 加载CSV数据
print('加载台风路径数据...')
print('=' * 60)

df = pd.read_csv(CSV_PATH)

# 基本统计信息
print(f'数据形状: {df.shape}')
print(f'\n列名:')
for col in df.columns:
    print(f'  - {col}')

print(f'\n数据类型:')
print(df.dtypes)

print(f'\n缺失值统计:')
print(df.isnull().sum())

print(f'\n前5行数据:')
df.head()

In [ ]:
# 数据基本统计
print('数据基本统计信息:')
print('=' * 60)

# 数值列统计
numeric_cols = ['latitude', 'longitude', 'center_pressure', 'max_wind_speed', 'moving_speed']
print(df[numeric_cols].describe())

# 台风数量统计
print(f'\n台风数量统计:')
print(f'  唯一台风编号数: {df["ty_code"].nunique()}')
print(f'  唯一台风名称数: {df["ty_name_en"].nunique()}')

# 年份范围
df['year'] = pd.to_datetime(df['timestamp']).dt.year
print(f'\n年份范围: {df["year"].min()} - {df["year"].max()}')
print(f'  共 {df["year"].nunique()} 年数据')

### 2.2 数据可视化探索

In [ ]:
# 台风路径分布可视化
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 用来显示负号
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. 经纬度分布
ax1 = axes[0, 0]
scatter = ax1.scatter(df['longitude'], df['latitude'], 
                     c=df['center_pressure'], cmap='viridis_r', 
                     alpha=0.5, s=10)
ax1.set_xlabel('经度 (°)', fontsize=12)
ax1.set_ylabel('纬度 (°)', fontsize=12)
ax1.set_title('台风路径分布（按气压着色）', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
cbar1 = plt.colorbar(scatter, ax=ax1)
cbar1.set_label('中心气压 (hPa)', fontsize=10)

# 2. 中心气压分布
ax2 = axes[0, 1]
ax2.hist(df['center_pressure'].dropna(), bins=50, color='skyblue', edgecolor='black', alpha=0.7)
ax2.set_xlabel('中心气压 (hPa)', fontsize=12)
ax2.set_ylabel('频数', fontsize=12)
ax2.set_title('中心气压分布', fontsize=14, fontweight='bold')
mean_pressure = df['center_pressure'].mean()
ax2.axvline(mean_pressure, color='red', linestyle='--', 
           label=f'均值: {mean_pressure:.1f}')
ax2.legend()

# 3. 最大风速分布
ax3 = axes[1, 0]
ax3.hist(df['max_wind_speed'].dropna(), bins=50, color='lightcoral', edgecolor='black', alpha=0.7)
ax3.set_xlabel('最大风速 (m/s)', fontsize=12)
ax3.set_ylabel('频数', fontsize=12)
ax3.set_title('最大风速分布', fontsize=14, fontweight='bold')
mean_wind = df['max_wind_speed'].mean()
ax3.axvline(mean_wind, color='blue', linestyle='--',
           label=f'均值: {mean_wind:.1f}')
ax3.legend()

# 4. 年度台风数量趋势
ax4 = axes[1, 1]
yearly_counts = df.groupby('year')['ty_code'].nunique()
ax4.plot(yearly_counts.index, yearly_counts.values, marker='o', linewidth=2, markersize=4)
ax4.set_xlabel('年份', fontsize=12)
ax4.set_ylabel('台风数量', fontsize=12)
ax4.set_title('年度台风数量趋势', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'data_exploration.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'✓ 数据探索图表已保存到: {RESULTS_DIR / "data_exploration.png"}')

---

## 3. 模型架构定义 <a id='section3'></a>

### 3.1 数据预处理器

In [ ]:
# 特征列定义
FEATURE_COLUMNS = [
    'latitude', 'longitude', 'center_pressure', 'max_wind_speed',
    'moving_speed', 'moving_direction', 'hour', 'month',
    'velocity_lat', 'velocity_lon', 'acceleration_lat', 'acceleration_lon',
    'month_sin', 'month_cos'
]

class NormalizationParams:
    '''归一化参数'''
    def __init__(self):
        # 经纬度使用Min-Max归一化
        self.lat_min, self.lat_max = -90.0, 90.0
        self.lon_min, self.lon_max = -180.0, 180.0
        
        # 其他特征使用Z-Score标准化
        self.pressure_mean, self.pressure_std = 1000.0, 50.0
        self.wind_mean, self.wind_std = 20.0, 15.0
        self.speed_mean, self.speed_std = 15.0, 10.0


class DataPreprocessor:
    '''数据预处理器'''
    
    def __init__(self, sequence_length=12, prediction_steps=8):
        self.sequence_length = sequence_length
        self.prediction_steps = prediction_steps
        self.norm_params = NormalizationParams()
    
    def clean_data(self, paths):
        '''数据清洗'''
        # 按时间排序
        paths = sorted(paths, key=lambda x: x.timestamp if hasattr(x, 'timestamp') else x['timestamp'])
        
        # 去重
        seen = set()
        cleaned = []
        for p in paths:
            ts = p.timestamp if hasattr(p, 'timestamp') else p['timestamp']
            if ts not in seen:
                seen.add(ts)
                cleaned.append(p)
        
        return cleaned
    
    def _safe_float(self, value, default=0.0):
        '''安全转换为浮点数，处理空字符串、NaN、None等'''
        if value is None:
            return default
        if isinstance(value, str):
            value = value.strip()
            if value == '' or value.lower() in ('nan', 'none', 'null'):
                return default
            try:
                return float(value)
            except ValueError:
                return default
        try:
            result = float(value)
            if np.isnan(result) or np.isinf(result):
                return default
            return result
        except (TypeError, ValueError):
            return default
    
    def extract_features(self, paths):
        '''特征提取'''
        features = []
        
        for i, p in enumerate(paths):
            # 基础特征 - 使用安全转换
            lat = self._safe_float(p.latitude if hasattr(p, 'latitude') else p['latitude'], 0.0)
            lon = self._safe_float(p.longitude if hasattr(p, 'longitude') else p['longitude'], 0.0)
            pressure = self._safe_float(p.center_pressure if hasattr(p, 'center_pressure') else p.get('center_pressure'), 1000.0)
            wind = self._safe_float(p.max_wind_speed if hasattr(p, 'max_wind_speed') else p.get('max_wind_speed'), 20.0)
            speed = self._safe_float(p.moving_speed if hasattr(p, 'moving_speed') else p.get('moving_speed'), 15.0)
            direction = self._safe_float(p.moving_direction if hasattr(p, 'moving_direction') else p.get('moving_direction'), 0.0)
            ts = pd.to_datetime(p.timestamp if hasattr(p, 'timestamp') else p['timestamp'])
            
            # 时间特征
            hour = ts.hour
            month = ts.month
            month_sin = np.sin(2 * np.pi * month / 12)
            month_cos = np.cos(2 * np.pi * month / 12)
            
            # 计算速度和加速度
            if i > 0:
                prev = paths[i-1]
                prev_lat = self._safe_float(prev.latitude if hasattr(prev, 'latitude') else prev['latitude'], lat)
                prev_lon = self._safe_float(prev.longitude if hasattr(prev, 'longitude') else prev['longitude'], lon)
                velocity_lat = lat - prev_lat
                velocity_lon = lon - prev_lon
            else:
                velocity_lat, velocity_lon = 0.0, 0.0
            
            if i > 1:
                prev = paths[i-1]
                prev_prev = paths[i-2]
                prev_lat = self._safe_float(prev.latitude if hasattr(prev, 'latitude') else prev['latitude'], lat)
                prev_lon = self._safe_float(prev.longitude if hasattr(prev, 'longitude') else prev['longitude'], lon)
                prev_prev_lat = self._safe_float(prev_prev.latitude if hasattr(prev_prev, 'latitude') else prev_prev['latitude'], lat)
                prev_prev_lon = self._safe_float(prev_prev.longitude if hasattr(prev_prev, 'longitude') else prev_prev['longitude'], lon)
                prev_vel_lat = prev_lat - prev_prev_lat
                prev_vel_lon = prev_lon - prev_prev_lon
                acceleration_lat = velocity_lat - prev_vel_lat
                acceleration_lon = velocity_lon - prev_vel_lon
            else:
                acceleration_lat, acceleration_lon = 0.0, 0.0
            
            features.append([
                lat, lon, pressure, wind,
                speed, direction, hour, month,
                velocity_lat, velocity_lon,
                acceleration_lat, acceleration_lon,
                month_sin, month_cos
            ])
        
        return np.array(features, dtype=np.float32)
    
    def normalize(self, features):
        '''归一化'''
        # 确保输入是浮点数数组
        if features.dtype != np.float32 and features.dtype != np.float64:
            features = features.astype(np.float32)
        
        normalized = features.copy().astype(np.float32)
        p = self.norm_params
        
        # Min-Max归一化
        normalized[:, 0] = (features[:, 0] - p.lat_min) / (p.lat_max - p.lat_min)
        normalized[:, 1] = (features[:, 1] - p.lon_min) / (p.lon_max - p.lon_min)
        normalized[:, 5] = features[:, 5] / 360.0  # 方向
        normalized[:, 6] = features[:, 6] / 23.0   # 小时
        normalized[:, 7] = (features[:, 7] - 1) / 11.0  # 月份
        
        # Z-Score标准化
        normalized[:, 2] = (features[:, 2] - p.pressure_mean) / p.pressure_std
        normalized[:, 3] = (features[:, 3] - p.wind_mean) / p.wind_std
        normalized[:, 4] = (features[:, 4] - p.speed_mean) / p.speed_std
        
        # 速度和加速度标准化
        normalized[:, 8:12] = features[:, 8:12] / 5.0
        
        return normalized
    
    def create_sequences(self, normalized_features):
        '''创建序列'''
        inputs, targets = [], []
        
        for i in range(len(normalized_features) - self.sequence_length - self.prediction_steps + 1):
            input_seq = normalized_features[i:i + self.sequence_length]
            target_seq = normalized_features[i + self.sequence_length:i + self.sequence_length + self.prediction_steps, :4]
            inputs.append(input_seq)
            targets.append(target_seq)
        
        return np.array(inputs, dtype=np.float32), np.array(targets, dtype=np.float32)

print('✓ 数据预处理器定义完成')
print(f'  特征维度: {len(FEATURE_COLUMNS)}')
print(f'  特征列表: {FEATURE_COLUMNS}')

### 3.2 PyTorch数据集

In [ ]:
class TyphoonPathData:
    '''台风路径数据类'''
    def __init__(self, data_dict):
        self.typhoon_id = data_dict.get('ty_code', '')
        self.timestamp = data_dict.get('timestamp', '')
        self.latitude = data_dict.get('latitude', 0.0)
        self.longitude = data_dict.get('longitude', 0.0)
        self.center_pressure = data_dict.get('center_pressure', 1000)
        self.max_wind_speed = data_dict.get('max_wind_speed', 20)
        self.moving_speed = data_dict.get('moving_speed', 15)
        self.moving_direction = data_dict.get('moving_direction', 0)


class CSVTyphoonDataset(Dataset):
    '''CSV台风数据集'''
    
    def __init__(self, csv_path=None, sequence_length=12, prediction_steps=8, 
                 start_year=None, end_year=None):
        self.sequence_length = sequence_length
        self.prediction_steps = prediction_steps
        self.preprocessor = DataPreprocessor(sequence_length, prediction_steps)
        
        # 加载数据
        if csv_path is None:
            csv_path = CSV_PATH
        
        df = pd.read_csv(csv_path)
        df['year'] = pd.to_datetime(df['timestamp']).dt.year
        
        # 按年份过滤
        if start_year is not None:
            df = df[df['year'] >= start_year]
        if end_year is not None:
            df = df[df['year'] <= end_year]
        
        # 转换为TyphoonPathData对象
        typhoon_paths = [TyphoonPathData(row.to_dict()) for _, row in df.iterrows()]
        
        # 构建样本
        self.samples = self._build_samples(typhoon_paths)
        
        print(f'数据集初始化完成: {len(self.samples)} 个样本')
    
    def _build_samples(self, paths):
        '''构建训练样本'''
        if not paths:
            return []
        
        # 按台风ID分组
        grouped = defaultdict(list)
        for p in paths:
            grouped[p.typhoon_id].append(p)
        
        samples = []
        
        for typhoon_id, typhoon_paths in grouped.items():
            # 数据清洗
            cleaned = self.preprocessor.clean_data(typhoon_paths)
            
            if len(cleaned) < self.sequence_length + self.prediction_steps:
                continue
            
            # 特征提取
            features = self.preprocessor.extract_features(cleaned)
            
            # 归一化
            normalized = self.preprocessor.normalize(features)
            
            # 构建序列
            inputs, targets = self.preprocessor.create_sequences(normalized)
            
            for i in range(len(inputs)):
                samples.append((inputs[i], targets[i]))
        
        return samples
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        input_seq, target_seq = self.samples[idx]
        return torch.FloatTensor(input_seq), torch.FloatTensor(target_seq)


class TyphoonDataCollator:
    '''数据收集器'''
    def __call__(self, batch):
        inputs = [item[0] for item in batch]
        targets = [item[1] for item in batch]
        return torch.stack(inputs, dim=0), torch.stack(targets, dim=0)

print('✓ PyTorch数据集定义完成')

### 3.3 Transformer + LSTM 模型定义

In [ ]:
class TransformerLSTMModel(nn.Module):
    '''
    Transformer + LSTM 混合模型
    
    架构：
    1. LSTM编码器 - 提取时序特征
    2. Transformer编码器 - 建模长程依赖
    3. 多步预测头 - 独立预测每个时间步
    4. 不确定性估计 - 输出预测分布
    '''
    
    def __init__(
        self,
        input_size=14,
        hidden_size=256,
        num_lstm_layers=2,
        num_transformer_layers=2,
        num_heads=8,
        output_size=4,
        prediction_steps=8,
        dropout=0.2
    ):
        super().__init__()
        
        self.hidden_size = hidden_size
        self.prediction_steps = prediction_steps
        
        # LSTM编码器
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_lstm_layers,
            batch_first=True,
            dropout=dropout if num_lstm_layers > 1 else 0,
            bidirectional=False
        )
        
        # Transformer编码器
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=num_heads,
            dim_feedforward=hidden_size * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_transformer_layers)
        
        # Layer Normalization
        self.layer_norm1 = nn.LayerNorm(hidden_size)
        self.layer_norm2 = nn.LayerNorm(hidden_size)
        
        # 预测头 - 为每个时间步独立预测均值和方差
        self.prediction_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_size, hidden_size // 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_size // 2, output_size * 2)  # 均值和方差
            ) for _ in range(prediction_steps)
        ])
        
        # 置信度估计头
        self.confidence_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, prediction_steps),
            nn.Sigmoid()
        )
        
        self._init_weights()
    
    def _init_weights(self):
        '''初始化权重'''
        for name, param in self.lstm.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)
        
        for head in self.prediction_heads:
            for layer in head:
                if isinstance(layer, nn.Linear):
                    nn.init.xavier_uniform_(layer.weight)
                    nn.init.zeros_(layer.bias)
    
    def forward(self, x):
        '''
        前向传播
        
        Args:
            x: 输入序列 [batch, seq_len, input_size]
        
        Returns:
            predictions_mean: 预测均值 [batch, pred_steps, output_size]
            predictions_std: 预测标准差 [batch, pred_steps, output_size]
            confidence: 置信度 [batch, pred_steps]
        '''
        # LSTM编码
        lstm_out, _ = self.lstm(x)
        lstm_out = self.layer_norm1(lstm_out)
        
        # Transformer编码
        transformer_out = self.transformer(lstm_out)
        transformer_out = self.layer_norm2(transformer_out + lstm_out)  # 残差连接
        
        # 取最后时刻的上下文
        context = transformer_out[:, -1, :]
        
        # 多步预测（输出均值和方差）
        predictions_mean = []
        predictions_std = []
        
        for head in self.prediction_heads:
            output = head(context)
            mean = output[..., :4]
            std = F.softplus(output[..., 4:]) + 1e-6  # 确保标准差为正
            predictions_mean.append(mean)
            predictions_std.append(std)
        
        predictions_mean = torch.stack(predictions_mean, dim=1)
        predictions_std = torch.stack(predictions_std, dim=1)
        
        # 置信度估计
        confidence = self.confidence_head(context)
        
        return predictions_mean, predictions_std, confidence


# 测试模型结构
print('测试模型结构...')
test_model = TransformerLSTMModel(
    input_size=14,
    hidden_size=256,
    num_lstm_layers=2,
    num_transformer_layers=2,
    num_heads=8,
    output_size=4,
    prediction_steps=8,
    dropout=0.2
)

# 统计参数量
total_params = sum(p.numel() for p in test_model.parameters())
trainable_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)

print('=' * 60)
print('模型结构:')
print('=' * 60)
print(test_model)
print('\n模型参数统计:')
print(f'  总参数量: {total_params:,}')
print(f'  可训练参数量: {trainable_params:,}')

# 测试前向传播
test_input = torch.randn(2, 12, 14)  # [batch=2, seq_len=12, features=14]
with torch.no_grad():
    mean, std, conf = test_model(test_input)
    print(f'\n测试输入形状: {test_input.shape}')
    print(f'预测均值形状: {mean.shape}')
    print(f'预测标准差形状: {std.shape}')
    print(f'置信度形状: {conf.shape}')
    print('\n✓ 模型前向传播测试通过！')

### 3.4 增强版损失函数

In [ ]:
class EnhancedLoss(nn.Module):
    '''
    增强版损失函数
    
    包含：
    1. 负对数似然损失（概率预测）
    2. 物理约束损失
    3. 时序一致性损失
    4. 置信度校准损失
    '''
    
    def __init__(
        self,
        path_weight=1.0,
        intensity_weight=0.5,
        physics_weight=0.3,
        temporal_weight=0.2,
        confidence_weight=0.5
    ):
        super().__init__()
        self.path_weight = path_weight
        self.intensity_weight = intensity_weight
        self.physics_weight = physics_weight
        self.temporal_weight = temporal_weight
        self.confidence_weight = confidence_weight
    
    def forward(self, predictions_mean, predictions_std, confidence, targets):
        '''
        计算损失
        
        Args:
            predictions_mean: 预测均值 [batch, pred_steps, 4]
            predictions_std: 预测标准差 [batch, pred_steps, 4]
            confidence: 置信度 [batch, pred_steps]
            targets: 目标值 [batch, pred_steps, 4]
        '''
        # 1. 负对数似然损失
        nll_loss = 0.5 * torch.log(2 * np.pi * predictions_std ** 2) + \
                   (targets - predictions_mean) ** 2 / (2 * predictions_std ** 2)
        nll_loss = nll_loss.mean()
        
        # 2. 路径损失（经纬度）
        path_loss = torch.mean((predictions_mean[:, :, :2] - targets[:, :, :2]) ** 2)
        
        # 3. 强度损失（气压、风速）
        intensity_loss = torch.mean((predictions_mean[:, :, 2:] - targets[:, :, 2:]) ** 2)
        
        # 4. 物理约束损失 - 最大移动速度限制
        pred_lats = predictions_mean[:, :, 0]
        pred_lons = predictions_mean[:, :, 1]
        
        lat_diff = torch.diff(pred_lats, dim=1)
        lon_diff = torch.diff(pred_lons, dim=1)
        distance = torch.sqrt(lat_diff ** 2 + lon_diff ** 2)
        
        # 惩罚过大的移动（6小时最大移动5度，归一化后约为0.028）
        physics_loss = torch.mean(F.relu(distance - 0.028) ** 2)
        
        # 5. 时序一致性损失
        lat_velocity = torch.diff(pred_lats, dim=1)
        lon_velocity = torch.diff(pred_lons, dim=1)
        first_order_smooth = torch.mean(lat_velocity ** 2) + torch.mean(lon_velocity ** 2)
        
        if pred_lats.shape[1] > 2:
            lat_acceleration = torch.diff(lat_velocity, dim=1)
            lon_acceleration = torch.diff(lon_velocity, dim=1)
            second_order_smooth = torch.mean(lat_acceleration ** 2) + torch.mean(lon_acceleration ** 2)
        else:
            second_order_smooth = torch.tensor(0.0, device=pred_lats.device)
        
        temporal_loss = first_order_smooth + 0.5 * second_order_smooth
        
        # 6. 置信度校准损失
        with torch.no_grad():
            actual_error = torch.mean((predictions_mean[:, :, :2] - targets[:, :, :2]) ** 2, dim=2)
            target_confidence = torch.exp(-actual_error * 10)
        
        confidence_loss = F.mse_loss(confidence, target_confidence)
        
        # 总损失
        total_loss = (
            nll_loss +
            self.path_weight * path_loss +
            self.intensity_weight * intensity_loss +
            self.physics_weight * physics_loss +
            self.temporal_weight * temporal_loss +
            self.confidence_weight * confidence_loss
        )
        
        return total_loss, {
            'nll': nll_loss.item(),
            'path': path_loss.item(),
            'intensity': intensity_loss.item(),
            'physics': physics_loss.item(),
            'temporal': temporal_loss.item(),
            'confidence': confidence_loss.item(),
            'total': total_loss.item()
        }

print('✓ 增强版损失函数定义完成')
print('损失函数组成:')
print('  1. 负对数似然损失 (NLL)')
print('  2. 路径损失 (经纬度MSE)')
print('  3. 强度损失 (气压、风速MSE)')
print('  4. 物理约束损失 (移动速度限制)')
print('  5. 时序一致性损失 (平滑约束)')
print('  6. 置信度校准损失')

---

## 4. 模型训练 <a id='section4'></a>

### 4.1 训练配置参数

In [ ]:
# 训练配置参数
class TrainingConfig:
    '''训练配置'''
    # 数据参数
    START_YEAR = 2000
    END_YEAR = 2020
    SEQUENCE_LENGTH = 12  # 输入序列长度（过去72小时，每6小时一个点）
    PREDICTION_STEPS = 8  # 预测步数（未来48小时，每6小时一个点）
    
    # 模型参数
    INPUT_SIZE = 14
    HIDDEN_SIZE = 256
    NUM_LSTM_LAYERS = 2
    NUM_TRANSFORMER_LAYERS = 2
    NUM_HEADS = 8
    OUTPUT_SIZE = 4
    DROPOUT = 0.2
    
    # 训练参数
    BATCH_SIZE = 64
    EPOCHS = 100
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 1e-5
    VAL_SPLIT = 0.2
    EARLY_STOPPING_PATIENCE = 15
    
    # 设备
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # 保存路径
    SAVE_DIR = MODELS_DIR

# 显示配置
print('训练配置参数:')
print('=' * 60)
print(f'数据年份范围: {TrainingConfig.START_YEAR} - {TrainingConfig.END_YEAR}')
print(f'输入序列长度: {TrainingConfig.SEQUENCE_LENGTH} 个时间点')
print(f'预测步数: {TrainingConfig.PREDICTION_STEPS} 个时间点')
print(f'\n模型参数:')
print(f'  输入维度: {TrainingConfig.INPUT_SIZE}')
print(f'  隐藏层维度: {TrainingConfig.HIDDEN_SIZE}')
print(f'  LSTM层数: {TrainingConfig.NUM_LSTM_LAYERS}')
print(f'  Transformer层数: {TrainingConfig.NUM_TRANSFORMER_LAYERS}')
print(f'  注意力头数: {TrainingConfig.NUM_HEADS}')
print(f'\n训练参数:')
print(f'  批次大小: {TrainingConfig.BATCH_SIZE}')
print(f'  训练轮数: {TrainingConfig.EPOCHS}')
print(f'  学习率: {TrainingConfig.LEARNING_RATE}')
print(f'  权重衰减: {TrainingConfig.WEIGHT_DECAY}')
print(f'  验证集比例: {TrainingConfig.VAL_SPLIT}')
print(f'  早停耐心值: {TrainingConfig.EARLY_STOPPING_PATIENCE}')
print(f'\n训练设备: {TrainingConfig.DEVICE}')
print(f'模型保存目录: {TrainingConfig.SAVE_DIR}')

### 4.2 加载数据集

In [ ]:
# 加载数据集
print('加载数据集...')
print('=' * 60)

full_dataset = CSVTyphoonDataset(
    csv_path=CSV_PATH,
    sequence_length=TrainingConfig.SEQUENCE_LENGTH,
    prediction_steps=TrainingConfig.PREDICTION_STEPS,
    start_year=TrainingConfig.START_YEAR,
    end_year=TrainingConfig.END_YEAR
)

print(f'\n数据集总大小: {len(full_dataset)} 个样本')

if len(full_dataset) == 0:
    raise ValueError('数据集为空，请检查数据路径和年份范围')

# 划分训练集和验证集
val_size = int(len(full_dataset) * TrainingConfig.VAL_SPLIT)
train_size = len(full_dataset) - val_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f'训练集大小: {len(train_dataset)} 个样本')
print(f'验证集大小: {len(val_dataset)} 个样本')

# 创建数据加载器
train_loader = DataLoader(
    train_dataset,
    batch_size=TrainingConfig.BATCH_SIZE,
    shuffle=True,
    collate_fn=TyphoonDataCollator(),
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=TrainingConfig.BATCH_SIZE,
    shuffle=False,
    collate_fn=TyphoonDataCollator(),
    num_workers=0
)

print(f'\n训练批次数量: {len(train_loader)}')
print(f'验证批次数量: {len(val_loader)}')

# 检查一个批次的数据
sample_inputs, sample_targets = next(iter(train_loader))
print(f'\n样本输入形状: {sample_inputs.shape}')
print(f'样本目标形状: {sample_targets.shape}')
print('\n✓ 数据集加载完成！')

### 4.3 初始化模型和优化器

In [ ]:
# 初始化模型
print('初始化模型...')
print('=' * 60)

model = TransformerLSTMModel(
    input_size=TrainingConfig.INPUT_SIZE,
    hidden_size=TrainingConfig.HIDDEN_SIZE,
    num_lstm_layers=TrainingConfig.NUM_LSTM_LAYERS,
    num_transformer_layers=TrainingConfig.NUM_TRANSFORMER_LAYERS,
    num_heads=TrainingConfig.NUM_HEADS,
    output_size=TrainingConfig.OUTPUT_SIZE,
    prediction_steps=TrainingConfig.PREDICTION_STEPS,
    dropout=TrainingConfig.DROPOUT
).to(TrainingConfig.DEVICE)

# 统计参数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'模型总参数量: {total_params:,}')
print(f'可训练参数量: {trainable_params:,}')

# 损失函数
criterion = EnhancedLoss(
    path_weight=1.0,
    intensity_weight=0.5,
    physics_weight=0.3,
    temporal_weight=0.2,
    confidence_weight=0.5
)

# 优化器
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=TrainingConfig.LEARNING_RATE,
    weight_decay=TrainingConfig.WEIGHT_DECAY
)

# 学习率调度器
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5
)

print('\n✓ 模型初始化完成！')
print(f'设备: {TrainingConfig.DEVICE}')

### 4.4 训练循环

In [ ]:
# 训练函数
def train_epoch(model, train_loader, criterion, optimizer, device):
    '''训练一个epoch'''
    model.train()
    total_loss = 0
    total_metrics = {
        'nll': 0, 'path': 0, 'intensity': 0,
        'physics': 0, 'temporal': 0, 'confidence': 0
    }
    
    pbar = tqdm(train_loader, desc='Training', leave=False)
    for batch_idx, (inputs, targets) in enumerate(pbar):
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()
        
        # 前向传播
        pred_mean, pred_std, confidence = model(inputs)
        
        # 计算损失
        loss, metrics = criterion(pred_mean, pred_std, confidence, targets)
        
        # 反向传播
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        for key in total_metrics:
            total_metrics[key] += metrics[key]
        
        pbar.set_postfix({'loss': f'{loss.item():.6f}'})
    
    avg_loss = total_loss / len(train_loader)
    avg_metrics = {k: v / len(train_loader) for k, v in total_metrics.items()}
    
    return avg_loss, avg_metrics


def validate(model, val_loader, criterion, device):
    '''验证模型'''
    model.eval()
    total_loss = 0
    all_predictions = []
    all_targets = []
    all_confidences = []
    
    with torch.no_grad():
        for inputs, targets in tqdm(val_loader, desc='Validation', leave=False):
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            pred_mean, pred_std, confidence = model(inputs)
            loss, _ = criterion(pred_mean, pred_std, confidence, targets)
            
            total_loss += loss.item()
            all_predictions.append(pred_mean.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
            all_confidences.append(confidence.cpu().numpy())
    
    avg_loss = total_loss / len(val_loader)
    
    # 计算评估指标
    predictions = np.concatenate(all_predictions, axis=0)
    targets = np.concatenate(all_targets, axis=0)
    confidences = np.concatenate(all_confidences, axis=0)
    
    mae_lat = np.mean(np.abs(predictions[:, :, 0] - targets[:, :, 0]))
    mae_lon = np.mean(np.abs(predictions[:, :, 1] - targets[:, :, 1]))
    mae_pressure = np.mean(np.abs(predictions[:, :, 2] - targets[:, :, 2]))
    mae_wind = np.mean(np.abs(predictions[:, :, 3] - targets[:, :, 3]))
    avg_confidence = np.mean(confidences)
    
    # 转换为实际度数
    mae_lat_deg = mae_lat * 180
    mae_lon_deg = mae_lon * 360
    
    return avg_loss, {
        'mae_lat': mae_lat,
        'mae_lon': mae_lon,
        'mae_lat_deg': mae_lat_deg,
        'mae_lon_deg': mae_lon_deg,
        'mae_pressure': mae_pressure,
        'mae_wind': mae_wind,
        'avg_confidence': avg_confidence
    }

print('✓ 训练函数定义完成')

### 4.5 执行训练（可配置快速/完整训练）

In [ ]:
# 训练模式选择
# 设置为True进行快速测试（5轮），False进行完整训练
QUICK_TEST = False

if QUICK_TEST:
    print('⚡ 快速测试模式')
    NUM_EPOCHS = 5
    EARLY_STOPPING_PATIENCE = 3
else:
    print('🔥 完整训练模式')
    NUM_EPOCHS = TrainingConfig.EPOCHS
    EARLY_STOPPING_PATIENCE = TrainingConfig.EARLY_STOPPING_PATIENCE

print(f'训练轮数: {NUM_EPOCHS}')
print(f'早停耐心值: {EARLY_STOPPING_PATIENCE}')

In [ ]:
# 训练历史记录
history = {
    'train_loss': [],
    'val_loss': [],
    'val_metrics': [],
    'learning_rate': []
}

best_val_loss = float('inf')
early_stopping_counter = 0

print('\n' + '=' * 70)
print('开始训练台风预测模型')
print('=' * 70)
print(f'训练样本数: {len(train_dataset)}')
print(f'验证样本数: {len(val_dataset)}')
print(f'总训练轮数: {NUM_EPOCHS}')
print(f'设备: {TrainingConfig.DEVICE}')
print('=' * 70)

# 训练循环
for epoch in range(NUM_EPOCHS):
    print(f'\nEpoch {epoch + 1}/{NUM_EPOCHS}')
    print('-' * 70)
    
    # 训练
    train_loss, train_metrics = train_epoch(
        model, train_loader, criterion, optimizer, TrainingConfig.DEVICE
    )
    
    # 验证
    val_loss, val_metrics = validate(
        model, val_loader, criterion, TrainingConfig.DEVICE
    )
    
    # 记录历史
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_metrics'].append(val_metrics)
    history['learning_rate'].append(optimizer.param_groups[0]['lr'])
    
    # 更新学习率
    scheduler.step(val_loss)
    
    # 打印结果
    print(f'训练损失: {train_loss:.6f}')
    print(f'验证损失: {val_loss:.6f}')
    print(f'验证指标:')
    print(f'  纬度误差: {val_metrics["mae_lat_deg"]:.2f}°')
    print(f'  经度误差: {val_metrics["mae_lon_deg"]:.2f}°')
    print(f'  气压误差: {val_metrics["mae_pressure"]:.4f}')
    print(f'  风速误差: {val_metrics["mae_wind"]:.4f}')
    print(f'  平均置信度: {val_metrics["avg_confidence"]:.4f}')
    
    # 保存最佳模型
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        early_stopping_counter = 0
        
        # 保存模型
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'epoch': epoch,
            'best_val_loss': best_val_loss,
            'history': history,
            'config': {
                'input_size': TrainingConfig.INPUT_SIZE,
                'hidden_size': TrainingConfig.HIDDEN_SIZE,
                'num_lstm_layers': TrainingConfig.NUM_LSTM_LAYERS,
                'num_transformer_layers': TrainingConfig.NUM_TRANSFORMER_LAYERS,
                'num_heads': TrainingConfig.NUM_HEADS,
                'output_size': TrainingConfig.OUTPUT_SIZE,
                'prediction_steps': TrainingConfig.PREDICTION_STEPS,
                'dropout': TrainingConfig.DROPOUT
            }
        }, TrainingConfig.SAVE_DIR / 'best_model.pth')
        print('✅ 保存最佳模型')
    else:
        early_stopping_counter += 1
    
    # 早停检查
    if early_stopping_counter >= EARLY_STOPPING_PATIENCE:
        print(f'\n⏹️ 早停触发，连续{EARLY_STOPPING_PATIENCE}轮未改善')
        break

print('\n' + '=' * 70)
print('训练完成！')
print(f'最佳验证损失: {best_val_loss:.6f}')
print('=' * 70)

---

## 5. 模型评估 <a id='section5'></a>

### 5.1 加载最佳模型

In [ ]:
# 加载最佳模型
print('加载最佳模型...')

checkpoint = torch.load(TrainingConfig.SAVE_DIR / 'best_model.pth', 
                       map_location=TrainingConfig.DEVICE,
                       weights_only=False)

# 重新创建模型
config = checkpoint['config']
best_model = TransformerLSTMModel(**config).to(TrainingConfig.DEVICE)
best_model.load_state_dict(checkpoint['model_state_dict'])
best_model.eval()

print('✓ 最佳模型加载成功！')
print(f'  训练轮次: {checkpoint["epoch"] + 1}')
print(f'  最佳验证损失: {checkpoint["best_val_loss"]:.6f}')

### 5.2 详细评估指标计算

In [ ]:
# 详细评估
def detailed_evaluation(model, data_loader, device):
    '''详细评估模型性能'''
    model.eval()
    
    all_predictions = []
    all_targets = []
    all_confidences = []
    
    with torch.no_grad():
        for inputs, targets in tqdm(data_loader, desc='评估中'):
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            pred_mean, pred_std, confidence = model(inputs)
            
            all_predictions.append(pred_mean.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
            all_confidences.append(confidence.cpu().numpy())
    
    # 合并所有批次
    predictions = np.concatenate(all_predictions, axis=0)
    targets = np.concatenate(all_targets, axis=0)
    confidences = np.concatenate(all_confidences, axis=0)
    
    # 计算各项误差指标
    # 纬度误差
    lat_errors = np.abs(predictions[:, :, 0] - targets[:, :, 0])
    lat_mae = np.mean(lat_errors)
    lat_rmse = np.sqrt(np.mean(lat_errors ** 2))
    
    # 经度误差
    lon_errors = np.abs(predictions[:, :, 1] - targets[:, :, 1])
    lon_mae = np.mean(lon_errors)
    lon_rmse = np.sqrt(np.mean(lon_errors ** 2))
    
    # 气压误差
    pressure_errors = np.abs(predictions[:, :, 2] - targets[:, :, 2])
    pressure_mae = np.mean(pressure_errors)
    pressure_rmse = np.sqrt(np.mean(pressure_errors ** 2))
    
    # 风速误差
    wind_errors = np.abs(predictions[:, :, 3] - targets[:, :, 3])
    wind_mae = np.mean(wind_errors)
    wind_rmse = np.sqrt(np.mean(wind_errors ** 2))
    
    # 路径误差（欧氏距离）
    path_errors = np.sqrt(lat_errors ** 2 + lon_errors ** 2)
    path_mae = np.mean(path_errors)
    path_rmse = np.sqrt(np.mean(path_errors ** 2))
    
    # 按预测时间步分析
    time_step_errors = []
    for t in range(predictions.shape[1]):
        step_lat_error = np.mean(lat_errors[:, t]) * 180  # 转换为度数
        step_lon_error = np.mean(lon_errors[:, t]) * 360
        step_path_error = np.mean(path_errors[:, t])
        time_step_errors.append({
            'hour': (t + 1) * 6,
            'lat_error': step_lat_error,
            'lon_error': step_lon_error,
            'path_error': step_path_error
        })
    
    # 置信度统计
    avg_confidence = np.mean(confidences)
    confidence_std = np.std(confidences)
    
    return {
        'lat_mae': lat_mae * 180,  # 转换为度数
        'lat_rmse': lat_rmse * 180,
        'lon_mae': lon_mae * 360,
        'lon_rmse': lon_rmse * 360,
        'pressure_mae': pressure_mae,
        'pressure_rmse': pressure_rmse,
        'wind_mae': wind_mae,
        'wind_rmse': wind_rmse,
        'path_mae': path_mae,
        'path_rmse': path_rmse,
        'time_step_errors': time_step_errors,
        'avg_confidence': avg_confidence,
        'confidence_std': confidence_std,
        'predictions': predictions,
        'targets': targets,
        'confidences': confidences
    }

# 执行评估
print('执行详细评估...')
eval_results = detailed_evaluation(best_model, val_loader, TrainingConfig.DEVICE)

print('\n✓ 评估完成！')

In [ ]:
# 打印评估结果
print('\n' + '=' * 70)
print('模型评估结果')
print('=' * 70)

print('\n【路径预测误差】')
print(f'  纬度 MAE: {eval_results["lat_mae"]:.4f}°')
print(f'  纬度 RMSE: {eval_results["lat_rmse"]:.4f}°')
print(f'  经度 MAE: {eval_results["lon_mae"]:.4f}°')
print(f'  经度 RMSE: {eval_results["lon_rmse"]:.4f}°')
print(f'  路径 MAE: {eval_results["path_mae"]:.4f}°')
print(f'  路径 RMSE: {eval_results["path_rmse"]:.4f}°')

print('\n【强度预测误差】')
print(f'  气压 MAE: {eval_results["pressure_mae"]:.4f}')
print(f'  气压 RMSE: {eval_results["pressure_rmse"]:.4f}')
print(f'  风速 MAE: {eval_results["wind_mae"]:.4f} m/s')
print(f'  风速 RMSE: {eval_results["wind_rmse"]:.4f} m/s')

print('\n【按预测时间步的路径误差】')
for error in eval_results['time_step_errors']:
    print(f"  {error['hour']:2d}h: {error['path_error']:.4f}° "
          f"(lat={error['lat_error']:.2f}°, lon={error['lon_error']:.2f}°)")

print('\n【置信度统计】')
print(f'  平均置信度: {eval_results["avg_confidence"]:.4f}')
print(f'  置信度标准差: {eval_results["confidence_std"]:.4f}')

print('\n' + '=' * 70)

# 保存评估结果
eval_file = RESULTS_DIR / 'evaluation_results.json'

# 转换numpy类型为Python原生类型
def convert_to_native(obj):
    if isinstance(obj, dict):
        return {k: convert_to_native(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_native(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

eval_data = {
    'lat_mae': float(eval_results['lat_mae']),
    'lat_rmse': float(eval_results['lat_rmse']),
    'lon_mae': float(eval_results['lon_mae']),
    'lon_rmse': float(eval_results['lon_rmse']),
    'pressure_mae': float(eval_results['pressure_mae']),
    'wind_mae': float(eval_results['wind_mae']),
    'path_mae': float(eval_results['path_mae']),
    'avg_confidence': float(eval_results['avg_confidence']),
    'time_step_errors': convert_to_native(eval_results['time_step_errors'])
}

with open(eval_file, 'w', encoding='utf-8') as f:
    json.dump(eval_data, f, indent=2, ensure_ascii=False)

print(f'✓ 评估结果已保存到: {eval_file}')

---

## 6. 结果可视化 <a id='section6'></a>

### 6.1 训练历史可视化

In [ ]:
# 训练历史可视化
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 用来显示负号
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# 1. 损失曲线
ax1 = axes[0, 0]
epochs_range = range(1, len(history['train_loss']) + 1)
train_line = ax1.plot(epochs_range, history['train_loss'], 'b-o', linewidth=2, markersize=6, label='训练损失')
val_line = ax1.plot(epochs_range, history['val_loss'], 'r-s', linewidth=2, markersize=6, label='验证损失')
# 添加数据标签
for i, (x, y) in enumerate(zip(epochs_range, history['train_loss'])):
    if i % max(1, len(epochs_range)//10) == 0:  # 每隔一定间隔显示标签
        ax1.annotate(f'{y:.4f}', (x, y), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=8, color='blue')
for i, (x, y) in enumerate(zip(epochs_range, history['val_loss'])):
    if i % max(1, len(epochs_range)//10) == 0:
        ax1.annotate(f'{y:.4f}', (x, y), textcoords='offset points', xytext=(0, -15), ha='center', fontsize=8, color='red')
ax1.set_xlabel('轮次', fontsize=12)
ax1.set_ylabel('损失', fontsize=12)
ax1.set_title('训练与验证损失曲线', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# 2. 学习率变化
ax2 = axes[0, 1]
ax2.plot(epochs_range, history['learning_rate'], 'g-o', linewidth=2, markersize=6)
# 添加数据标签
for i, (x, y) in enumerate(zip(epochs_range, history['learning_rate'])):
    if i % max(1, len(epochs_range)//10) == 0:
        ax2.annotate(f'{y:.2e}', (x, y), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=8, color='green')
ax2.set_xlabel('轮次', fontsize=12)
ax2.set_ylabel('学习率', fontsize=12)
ax2.set_title('学习率变化曲线', fontsize=14, fontweight='bold')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

# 3. 验证指标 - 路径误差
ax3 = axes[1, 0]
mae_lat = [m['mae_lat_deg'] for m in history['val_metrics']]
mae_lon = [m['mae_lon_deg'] for m in history['val_metrics']]
ax3.plot(epochs_range, mae_lat, 'b-o', linewidth=2, markersize=6, label='纬度误差')
ax3.plot(epochs_range, mae_lon, 'r-s', linewidth=2, markersize=6, label='经度误差')
# 添加数据标签
for i, (x, y) in enumerate(zip(epochs_range, mae_lat)):
    if i % max(1, len(epochs_range)//10) == 0:
        ax3.annotate(f'{y:.2f}', (x, y), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=8, color='blue')
for i, (x, y) in enumerate(zip(epochs_range, mae_lon)):
    if i % max(1, len(epochs_range)//10) == 0:
        ax3.annotate(f'{y:.2f}', (x, y), textcoords='offset points', xytext=(0, -15), ha='center', fontsize=8, color='red')
ax3.set_xlabel('轮次', fontsize=12)
ax3.set_ylabel('误差 (度)', fontsize=12)
ax3.set_title('验证集路径误差', fontsize=14, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

# 4. 验证指标 - 强度误差
ax4 = axes[1, 1]
mae_pressure = [m['mae_pressure'] for m in history['val_metrics']]
mae_wind = [m['mae_wind'] for m in history['val_metrics']]
ax4.plot(epochs_range, mae_pressure, 'b-o', linewidth=2, markersize=6, label='气压误差')
ax4.plot(epochs_range, mae_wind, 'r-s', linewidth=2, markersize=6, label='风速误差')
# 添加数据标签
for i, (x, y) in enumerate(zip(epochs_range, mae_pressure)):
    if i % max(1, len(epochs_range)//10) == 0:
        ax4.annotate(f'{y:.4f}', (x, y), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=8, color='blue')
for i, (x, y) in enumerate(zip(epochs_range, mae_wind)):
    if i % max(1, len(epochs_range)//10) == 0:
        ax4.annotate(f'{y:.4f}', (x, y), textcoords='offset points', xytext=(0, -15), ha='center', fontsize=8, color='red')
ax4.set_xlabel('轮次', fontsize=12)
ax4.set_ylabel('误差', fontsize=12)
ax4.set_title('验证集强度误差', fontsize=14, fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'✓ 训练历史图表已保存到: {RESULTS_DIR / "training_history.png"}')

### 6.2 按时间步误差分析

In [ ]:
# 按时间步误差分析
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 用来显示负号
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

hours = [e['hour'] for e in eval_results['time_step_errors']]
lat_errors = [e['lat_error'] for e in eval_results['time_step_errors']]
lon_errors = [e['lon_error'] for e in eval_results['time_step_errors']]
path_errors = [e['path_error'] for e in eval_results['time_step_errors']]

# 1. 经纬度误差
ax1 = axes[0]
ax1.plot(hours, lat_errors, 'b-o', linewidth=2, markersize=10, label='纬度误差')
ax1.plot(hours, lon_errors, 'r-s', linewidth=2, markersize=10, label='经度误差')
# 添加数据标签
for x, y in zip(hours, lat_errors):
    ax1.annotate(f'{y:.3f}', (x, y), textcoords='offset points', xytext=(0, 12), ha='center', fontsize=9, color='blue', fontweight='bold')
for x, y in zip(hours, lon_errors):
    ax1.annotate(f'{y:.3f}', (x, y), textcoords='offset points', xytext=(0, -18), ha='center', fontsize=9, color='red', fontweight='bold')
ax1.set_xlabel('预测时间 (小时)', fontsize=12)
ax1.set_ylabel('平均绝对误差 (度)', fontsize=12)
ax1.set_title('按预测时间步的经纬度误差', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# 2. 路径误差
ax2 = axes[1]
colors = plt.cm.viridis(np.linspace(0, 1, len(hours)))
bars = ax2.bar(range(len(hours)), path_errors, color=colors, edgecolor='black')
ax2.set_xlabel('预测时间 (小时)', fontsize=12)
ax2.set_ylabel('路径平均绝对误差 (度)', fontsize=12)
ax2.set_title('按预测时间步的路径误差', fontsize=14, fontweight='bold')
ax2.set_xticks(range(len(hours)))
ax2.set_xticklabels([f'{h}h' for h in hours])
ax2.grid(True, alpha=0.3, axis='y')

# 在柱状图上添加数值标签
for bar, error in zip(bars, path_errors):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{error:.3f}°', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'time_step_errors.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'✓ 时间步误差分析图表已保存到: {RESULTS_DIR / "time_step_errors.png"}')

### 6.3 预测结果散点图

In [ ]:
# 预测vs真实值散点图
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 用来显示负号
predictions = eval_results['predictions']
targets = eval_results['targets']

# 选择第一个预测时间步进行可视化
pred_t = predictions[:, 0, :]  # [batch, features]
target_t = targets[:, 0, :]

# 反归一化
pred_lat = pred_t[:, 0] * 180 - 90
pred_lon = pred_t[:, 1] * 360 - 180
target_lat = target_t[:, 0] * 180 - 90
target_lon = target_t[:, 1] * 360 - 180

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# 1. 纬度预测vs真实
ax1 = axes[0, 0]
ax1.scatter(target_lat, pred_lat, alpha=0.5, s=20, c='blue', edgecolors='none')
min_val = min(target_lat.min(), pred_lat.min())
max_val = max(target_lat.max(), pred_lat.max())
ax1.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='理想线')
ax1.set_xlabel('真实纬度 (°)', fontsize=12)
ax1.set_ylabel('预测纬度 (°)', fontsize=12)
ax1.set_title('纬度预测 vs 真实值', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. 经度预测vs真实
ax2 = axes[0, 1]
ax2.scatter(target_lon, pred_lon, alpha=0.5, s=20, c='green', edgecolors='none')
min_val = min(target_lon.min(), pred_lon.min())
max_val = max(target_lon.max(), pred_lon.max())
ax2.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='理想线')
ax2.set_xlabel('真实经度 (°)', fontsize=12)
ax2.set_ylabel('预测经度 (°)', fontsize=12)
ax2.set_title('经度预测 vs 真实值', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. 误差分布
ax3 = axes[1, 0]
lat_error = pred_lat - target_lat
lon_error = pred_lon - target_lon
ax3.hist(lat_error, bins=50, alpha=0.6, color='blue', label='纬度误差', edgecolor='black')
ax3.hist(lon_error, bins=50, alpha=0.6, color='green', label='经度误差', edgecolor='black')
ax3.axvline(0, color='red', linestyle='--', linewidth=2)
ax3.set_xlabel('误差 (度)', fontsize=12)
ax3.set_ylabel('频数', fontsize=12)
ax3.set_title('预测误差分布', fontsize=14, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. 路径可视化（随机选择10个样本）
ax4 = axes[1, 1]
n_samples = min(10, len(predictions))
indices = np.random.choice(len(predictions), n_samples, replace=False)

for idx in indices:
    # 真实路径
    true_lats = targets[idx, :, 0] * 180 - 90
    true_lons = targets[idx, :, 1] * 360 - 180
    
    # 预测路径
    pred_lats = predictions[idx, :, 0] * 180 - 90
    pred_lons = predictions[idx, :, 1] * 360 - 180
    
    ax4.plot(true_lons, true_lats, 'b-o', alpha=0.5, markersize=4, linewidth=1)
    ax4.plot(pred_lons, pred_lats, 'r--s', alpha=0.5, markersize=4, linewidth=1)

ax4.set_xlabel('经度 (°)', fontsize=12)
ax4.set_ylabel('纬度 (°)', fontsize=12)
ax4.set_title('台风路径预测可视化（随机样本）', fontsize=14, fontweight='bold')

# 添加图例
true_line = Line2D([0], [0], color='blue', marker='o', label='真实路径')
pred_line = Line2D([0], [0], color='red', marker='s', linestyle='--', label='预测路径')
ax4.legend(handles=[true_line, pred_line])
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'prediction_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'✓ 预测分析图表已保存到: {RESULTS_DIR / "prediction_analysis.png"}')

### 6.4 置信度分析

In [ ]:
# 置信度分析
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 用来显示负号
confidences = eval_results['confidences']

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 1. 置信度分布
ax1 = axes[0]
n, bins, patches = ax1.hist(confidences.flatten(), bins=50, color='skyblue', edgecolor='black', alpha=0.7)
mean_conf = np.mean(confidences)
ax1.axvline(mean_conf, color='red', linestyle='--', linewidth=2,
           label=f'均值: {mean_conf:.3f}')
# 添加均值标签
ax1.annotate(f'均值={mean_conf:.3f}', xy=(mean_conf, max(n)*0.9), 
            fontsize=11, color='red', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))
ax1.set_xlabel('置信度', fontsize=12)
ax1.set_ylabel('频数', fontsize=12)
ax1.set_title('置信度分布', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. 按时间步的置信度
ax2 = axes[1]
conf_by_step = np.mean(confidences, axis=0)
hours = [(i + 1) * 6 for i in range(len(conf_by_step))]
ax2.plot(hours, conf_by_step, 'g-o', linewidth=2, markersize=10)
# 添加数据标签
for x, y in zip(hours, conf_by_step):
    ax2.annotate(f'{y:.3f}', (x, y), textcoords='offset points', xytext=(0, 12), 
                ha='center', fontsize=9, color='green', fontweight='bold')
ax2.set_xlabel('预测时间 (小时)', fontsize=12)
ax2.set_ylabel('平均置信度', fontsize=12)
ax2.set_title('按预测时间步的置信度', fontsize=14, fontweight='bold')
ax2.set_ylim([0, 1.1])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confidence_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'✓ 置信度分析图表已保存到: {RESULTS_DIR / "confidence_analysis.png"}')

---

## 7. 模型保存与导出 <a id='section7'></a>

### 7.1 保存训练结果

In [ ]:
# 保存完整训练历史
history_file = RESULTS_DIR / 'training_history.json'

# 转换numpy类型为Python原生类型
def convert_to_native(obj):
    if isinstance(obj, dict):
        return {k: convert_to_native(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_native(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

history_native = convert_to_native(history)

with open(history_file, 'w', encoding='utf-8') as f:
    json.dump(history_native, f, indent=2, ensure_ascii=False)

print(f'✓ 训练历史已保存到: {history_file}')

# 保存最终模型
final_model_path = TrainingConfig.SAVE_DIR / 'final_model.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'history': history,
    'config': checkpoint['config']
}, final_model_path)

print(f'✓ 最终模型已保存到: {final_model_path}')

# 列出所有保存的文件
print('\n保存的文件列表:')
print('=' * 60)
print('\n模型文件:')
for f in TrainingConfig.SAVE_DIR.iterdir():
    size = f.stat().st_size / 1024 / 1024
    print(f'  {f.name}: {size:.2f} MB')

print('\n结果文件:')
for f in RESULTS_DIR.iterdir():
    if f.is_file():
        size = f.stat().st_size / 1024
        print(f'  {f.name}: {size:.2f} KB')

### 7.2 生成训练报告

In [ ]:
# 生成训练报告
report_lines = [
    '# 台风预测模型训练报告\n',
    '\n',
    '## 基本信息\n',
    f"- **训练时间**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n",
    f"- **模型架构**: Transformer + LSTM 混合神经网络\n",
    f"- **训练设备**: {TrainingConfig.DEVICE}\n",
    '\n',
    '## 数据配置\n',
    f"- **年份范围**: {TrainingConfig.START_YEAR} - {TrainingConfig.END_YEAR}\n",
    f"- **输入序列长度**: {TrainingConfig.SEQUENCE_LENGTH} 个时间点（过去72小时）\n",
    f"- **预测步数**: {TrainingConfig.PREDICTION_STEPS} 个时间点（未来48小时）\n",
    f"- **训练样本数**: {len(train_dataset)}\n",
    f"- **验证样本数**: {len(val_dataset)}\n",
    '\n',
    '## 模型参数\n',
    f"- **输入维度**: {TrainingConfig.INPUT_SIZE}\n",
    f"- **隐藏层维度**: {TrainingConfig.HIDDEN_SIZE}\n",
    f"- **LSTM层数**: {TrainingConfig.NUM_LSTM_LAYERS}\n",
    f"- **Transformer层数**: {TrainingConfig.NUM_TRANSFORMER_LAYERS}\n",
    f"- **注意力头数**: {TrainingConfig.NUM_HEADS}\n",
    f"- **总参数量**: {total_params:,}\n",
    '\n',
    '## 训练参数\n',
    f"- **批次大小**: {TrainingConfig.BATCH_SIZE}\n",
    f"- **训练轮数**: {len(history['train_loss'])} / {NUM_EPOCHS}\n",
    f"- **初始学习率**: {TrainingConfig.LEARNING_RATE}\n",
    f"- **权重衰减**: {TrainingConfig.WEIGHT_DECAY}\n",
    '\n',
    '## 评估结果\n',
    '\n',
    '### 路径预测误差\n',
    '| 指标 | 数值 |\n',
    '|------|------|\n',
    f"| 纬度 MAE | {eval_results['lat_mae']:.4f}° |\n",
    f"| 纬度 RMSE | {eval_results['lat_rmse']:.4f}° |\n",
    f"| 经度 MAE | {eval_results['lon_mae']:.4f}° |\n",
    f"| 经度 RMSE | {eval_results['lon_rmse']:.4f}° |\n",
    f"| 路径 MAE | {eval_results['path_mae']:.4f}° |\n",
    f"| 路径 RMSE | {eval_results['path_rmse']:.4f}° |\n",
    '\n',
    '### 强度预测误差\n',
    '| 指标 | 数值 |\n',
    '|------|------|\n',
    f"| 气压 MAE | {eval_results['pressure_mae']:.4f} |\n",
    f"| 气压 RMSE | {eval_results['pressure_rmse']:.4f} |\n",
    f"| 风速 MAE | {eval_results['wind_mae']:.4f} m/s |\n",
    f"| 风速 RMSE | {eval_results['wind_rmse']:.4f} m/s |\n",
    '\n',
    '### 按预测时间步的路径误差\n',
    '| 预测时间 | 路径误差 |\n',
    '|----------|----------|\n',
]

for error in eval_results['time_step_errors']:
    report_lines.append(f"| {error['hour']}h | {error['path_error']:.4f}° |\n")

report_lines.extend([
    '\n',
    '### 置信度统计\n',
    f"- **平均置信度**: {eval_results['avg_confidence']:.4f}\n",
    f"- **置信度标准差**: {eval_results['confidence_std']:.4f}\n",
    '\n',
    '## 文件输出\n',
    f"- **最佳模型**: {TrainingConfig.SAVE_DIR / 'best_model.pth'}\n",
    f"- **最终模型**: {final_model_path}\n",
    f"- **训练历史**: {history_file}\n",
    f"- **评估结果**: {eval_file}\n",
    '\n',
    '## 可视化图表\n',
    f"1. 数据探索: {RESULTS_DIR / 'data_exploration.png'}\n",
    f"2. 训练历史: {RESULTS_DIR / 'training_history.png'}\n",
    f"3. 时间步误差: {RESULTS_DIR / 'time_step_errors.png'}\n",
    f"4. 预测分析: {RESULTS_DIR / 'prediction_analysis.png'}\n",
    f"5. 置信度分析: {RESULTS_DIR / 'confidence_analysis.png'}\n",
])

report = ''.join(report_lines)

# 保存报告
report_file = RESULTS_DIR / 'training_report.md'
with open(report_file, 'w', encoding='utf-8') as f:
    f.write(report)

print(f'✓ 训练报告已保存到: {report_file}')
print('\n' + '=' * 70)
print('训练完成！所有文件已保存。')
print('=' * 70)

---

## 附录：命令行训练方式

除了使用本Notebook进行训练外，也可以使用命令行脚本进行训练：

### 快速测试
```bash
cd backend/training
python train_model_v3.py \
    --start-year 2018 \
    --end-year 2020 \
    --batch-size 32 \
    --epochs 20 \
    --early-stopping 5
```

### 标准训练
```bash
cd backend/training
python train_model_v3.py \
    --start-year 2000 \
    --end-year 2020 \
    --batch-size 64 \
    --epochs 100 \
    --lr 0.001 \
    --early-stopping 15
```

### 模型评估
```bash
cd backend/training
python evaluate_model.py \
    --model-path ./models/best_model.pth \
    --csv-path ../data/csv/typhoon_paths_1966_2026.csv \
    --start-year 2021 \
    --end-year 2024
```